<a href="https://colab.research.google.com/github/Terenceisstudying/SIT-UofG-QC-Assignment/blob/main/BB84-Attacker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BB84 Quantum Key Distribution with an Intercept-Resend Attacker

This notebook simulates true BB84 when an attacker is present to eavesdrop.

Showcasing 3 participants: Alice, Bob, and Charlie.

Alice and Bob are the legitimate participants. Charlie is not a legitimate participant. Charlie sits between Alice and Bob, intercepts qubits, measures them, and resends replacement qubits. Because Charlie does not know Alice's basis, his measurement can disturb the state that Bob receives. Alice and Bob detect that disturbance by publicly comparing only basis, testing a subset of the sifted bits, and aborting when the measured QBER is too high.


## Protocol overview

1. Alice generates random classical bits and random basis choices.
2. Alice encodes each bit as a qubit.
3. Charlie quantum-randomly decides whether to attack each qubit.
4. If Charlie attacks, he chooses a random basis, measures the qubit, and resends a new qubit matching his result and basis.
5. Bob chooses random bases and measures the qubits he receives.
6. Alice and Bob publicly compare bases only. Matching-basis positions become the sifted key.
7. Alice and Bob publicly reveal a quantum-random subset of sifted bits to estimate QBER.
8. If the QBER is above the threshold, they reject the channel. Otherwise, they keep the unrevealed sifted bits as the final demonstration key.


In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
from html import escape
from IPython.display import HTML, display


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 76.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=d2a16457cdf33a09990e42baeaa2d67c6f0208c5de0ae72f1cb30665224d8171
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


## Quantum randomness

To generate a random bit by applying a Hadamard gate to `|0>`, then measures it. The result is either `0` or `1` with equal probability.  
`|+> = (|0> + |1>) / sqrt(2)`

In [2]:
# Creates random Quantum bit.
# To generate genuinely random numbers, we need a physical process that is random.
# Applying a Hadamard gate
# For example: construct |+> and measure it.

def random_quantum_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)

    backend = BasicSimulator()
    compiled_circuit = transpile(qc, backend)
    job = backend.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    return int(list(counts.keys())[0])


# List of independent qubits.
def random_quantum_bits(n):
    return [random_quantum_bit() for _ in range(n)]

In [3]:
# Extra helper functions for getting random number with quantum measurement instead of random number generator.

def quantum_random_int(upper_bound):
    # Return a quantum-random integer from 0 to upper_bound - 1.
    if upper_bound <= 0:
        raise ValueError("upper_bound must be positive")

    bit_count = max(1, (upper_bound - 1).bit_length())

    # Rejection sampling avoids bias when upper_bound is not a power of 2.
    while True:
        value = 0
        for _ in range(bit_count):
            value = (value << 1) | random_quantum_bit()
        if value < upper_bound:
            return value


def quantum_random_float(precision_bits=8):
    # Approximate a random float in [0, 1) using quantum-random bits.
    value = 0
    for _ in range(precision_bits):
        value = (value << 1) | random_quantum_bit()
    return value / (2 ** precision_bits)

## Alice's encoding

Alice uses her bit and basis to prepare the corresponding BB84 state. Standard basis means `0` or `1`; diagonal basis means `+` or `-`.


In [4]:
# Alice encodes each bit
# bit|basis   |state
# 0  |standard| |0>
# 1  |standard| |1>
# 0  |diagonal| |+>
# 1  |diagonal| |->
def alice_encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)

    # A bit value of 1 starts from |1>; a bit value of 0 stays as |0>.
    if bit == 1:
        qc.x(0)

    # A diagonal-basis qubit is created by applying H after the bit is set.
    if basis == 1:
        qc.h(0)

    return qc

## Bob's measurement

Bob measures every qubit. If Bob uses the same basis as Alice and Charlie has not disturbed the qubit, Bob should recover Alice's bit. If Charlie measured in the wrong basis, Bob may get a different bit even when Bob and Alice used matching bases.


In [5]:
def bob_measure(prepared_qubit, basis):
    qc = prepared_qubit.copy()

    # if measuring in in diagonal basis (1), apply H
    if basis == 1:
        qc.h(0)
    # if measuring in standard basis (0), just do it
    qc.measure(0, 0)

    backend = BasicSimulator()
    compiled = transpile(qc, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    counts = result.get_counts(compiled)

    return int(list(counts.keys())[0])

## Charlie's intercept-resend attack

Charlie is the attacker. For each qubit, he uses quantum randomness to decide whether to attack. When he attacks, he chooses a random basis, measures the incoming qubit, and sends Bob a new qubit encoded with Charlie's measured bit and basis.

If Charlie chooses the wrong basis, his measurement can change the state. That disturbance can show up later as disagreements in Alice and Bob's matching-basis positions.


In [6]:
def charlie_intercept_resend(prepared_qubit, attack_probability):
    attack = quantum_random_float() < attack_probability

    if not attack:
        return prepared_qubit, {
            "attack": False,
            "basis": None,
            "bit": None,
            "resend": None,
        }

    charlie_basis = random_quantum_bit()
    charlie_bit = bob_measure(prepared_qubit, charlie_basis)
    resent_qubit = alice_encode_qubit(charlie_bit, charlie_basis)

    return resent_qubit, {
        "attack": True,
        "basis": charlie_basis,
        "bit": charlie_bit,
        "resend": qubit_label(charlie_bit, charlie_basis),
    }

## Lecture 3b table helpers

The table marks the matching-basis columns in red. Those red positions are the only positions that can enter the sifted key. Without disturbance, Alice and Bob should agree in every red column. Charlie's wrong-basis measurements can create disagreements in those red columns.


In [7]:
# Converting of the basis to 's' standard, 'd' diagonal to follow the lecture's convention
def basis_label(basis):
      if basis == 0:
          return "s"
      return "d"


# Converting of the qubit label to follow the lecture's convention
def qubit_label(bit, basis):
      if basis == 0:
          return str(bit)
      if bit == 0:
          return "+"
      return "-"


# Converting of Bob's bit that does not match Alice's basis to follow the lecture's convention
def bob_display_bit(i, alice_basis, bob_basis, bob_bits):
      if alice_basis[i] == bob_basis[i]:
          return str(bob_bits[i])
      return "?"


def maybe_basis_label(basis):
    if basis is None:
        return "?"
    return basis_label(basis)


def maybe_value(value):
    if value is None:
        return "?"
    return value


def table_cell(value, is_red=False):
    colour = "#c00000" if is_red else "#111111"
    weight = "700" if is_red else "400"
    return f"<td style='color:{colour}; font-weight:{weight}; padding:4px 8px; text-align:center;'>{escape(str(value))}</td>"


def display_bb84_attack_table(alice_bits, alice_basis, charlie_records, bob_basis, bob_bits, matching_positions):
    matching_set = set(matching_positions)
    n = len(alice_bits)

    rows = [
        ("Index", [i for i in range(n)], True),
        ("A bit", alice_bits, True),
        ("A basis", [basis_label(basis) for basis in alice_basis], True),
        ("A qubit", [qubit_label(alice_bits[i], alice_basis[i]) for i in range(n)], False),
        ("C attack", ["Y" if record["attack"] else "N" for record in charlie_records], False),
        ("C basis", [maybe_basis_label(record["basis"]) for record in charlie_records], False),
        ("C bit", [maybe_value(record["bit"]) for record in charlie_records], False),
        ("C resend", [maybe_value(record["resend"]) for record in charlie_records], False),
        ("B basis", [basis_label(basis) for basis in bob_basis], True),
        ("B bit", [bob_display_bit(i, alice_basis, bob_basis, bob_bits) for i in range(n)], True),
        ("B raw", bob_bits, False),
    ]

    html = """
    <style>
      .bb84-table { border-collapse: collapse; font-family: monospace; margin-top: 8px; }
      .bb84-table th { padding: 4px 10px; text-align: right; color: #111111; }
      .bb84-caption { font-family: sans-serif; margin-top: 8px; color: #333333; max-width: 1100px; }
    </style>
    <table class='bb84-table'>
    """

    for label, values, colour_matching in rows:
        html += f"<tr><th>{escape(label)}:</th>"
        for i, value in enumerate(values):
            html += table_cell(value, colour_matching and i in matching_set)
        html += "</tr>"

    html += "</table>"
    html += "<div class='bb84-caption'>Red columns are positions where Alice and Bob used the same basis. These should agree without disturbance. Charlie's wrong-basis measurements can cause disagreements in red columns.</div>"
    display(HTML(html))

## Sifting, QBER testing, and reconciliation

After Bob measures, Alice and Bob publicly compare basis choices only. They keep positions where the bases match.

They then reveal a quantum-random subset of those sifted positions to estimate the Quantum Bit Error Rate (QBER). Publicly revealed test bits cannot remain secret, so they are removed from the final key. If the QBER is too high, Alice and Bob abort because Charlie or channel noise has likely disturbed the transmission.

In [8]:
def choose_quantum_random_sample(items, sample_count):
    # Select sample_count items without using Python's random module.
    remaining = list(items)
    sample = []

    while remaining and len(sample) < sample_count:
        random_index = quantum_random_int(len(remaining))
        sample.append(remaining.pop(random_index))

    return sample


def estimate_qber(alice_bits, bob_bits, test_positions):
    if len(test_positions) == 0:
        return 0, 0.0

    error_count = 0
    for i in test_positions:
        if alice_bits[i] != bob_bits[i]:
            error_count += 1

    return error_count, error_count / len(test_positions)


def reconciliation_summary(final_alice_key, final_bob_key):
    mismatches = []
    for i in range(len(final_alice_key)):
        if final_alice_key[i] != final_bob_key[i]:
            mismatches.append(i)

    if len(mismatches) == 0:
        return "No reconciliation correction needed. The final keys already match.", mismatches

    return "Reconciliation would be needed here:", mismatches

## Run the attacker simulation

Used `n = 100` to make Charlie's disturbance easier to detect statistically. The default `attack_probability = 1.0` means Charlie attacks every qubit.

For comparison, set `attack_probability = 0.0` and rerun the notebook. With no attacker and no channel noise, QBER should be `0.0`, the channel should be accepted, and the final keys should match. With `attack_probability = 0.5`, QBER can be lower and may or may not exceed the threshold depending on the quantum-random sample.

In [9]:
# Constant that can be change as needed
n = 100
test_fraction = 0.5
# Quantum Bit Error Rate
qber_threshold = 0.2
attack_probability = 1.0


# Alice creates random bits and random bases.
alice_bits = random_quantum_bits(n)
alice_basis = random_quantum_bits(n)

# Alice prepares qubits, Charlie may intercept-resend, and Bob receives the resulting qubits.
qubits_for_bob = []
charlie_records = []
for i in range(n):
    prepared_qubit = alice_encode_qubit(alice_bits[i], alice_basis[i])
    transmitted_qubit, charlie_record = charlie_intercept_resend(prepared_qubit, attack_probability)
    qubits_for_bob.append(transmitted_qubit)
    charlie_records.append(charlie_record)

# Bob chooses random bases and measures every qubit.
bob_basis = random_quantum_bits(n)
bob_bits = []
for i in range(n):
    bob_bits.append(bob_measure(qubits_for_bob[i], bob_basis[i]))

# Alice and Bob publicly basis comparison (note that it is not bit comparison)
matching_positions = []
for i in range(n):
    if alice_basis[i] == bob_basis[i]:
        matching_positions.append(i)

# Alice and Bob keep only the bits where their basis matched
sifted_alice_key = [alice_bits[i] for i in matching_positions]
sifted_bob_key = [bob_bits[i] for i in matching_positions]

# Choose some sifted positions for public error-rate testing.
test_count = int(len(matching_positions) * test_fraction)
if len(matching_positions) > 0:
    test_count = max(1, test_count)
test_positions = choose_quantum_random_sample(matching_positions, test_count)
test_position_set = set(test_positions)

# Publicly revealed test bits are removed from the final secret key.
secret_positions = []
for i in matching_positions:
    if i not in test_position_set:
        secret_positions.append(i)

error_count, qber = estimate_qber(alice_bits, bob_bits, test_positions)
channel_accepted = qber <= qber_threshold

final_alice_key = [alice_bits[i] for i in secret_positions] if channel_accepted else []
final_bob_key = [bob_bits[i] for i in secret_positions] if channel_accepted else []
reconciliation_message, final_mismatches = reconciliation_summary(final_alice_key, final_bob_key)
attacked_count = sum(1 for record in charlie_records if record["attack"])

display_bb84_attack_table(alice_bits, alice_basis, charlie_records, bob_basis, bob_bits, matching_positions)

print("Bob raw measured bits:", bob_bits)
print("Charlie attacked positions:", [i for i, record in enumerate(charlie_records) if record["attack"]])
print("Charlie attacked count:", attacked_count)
print("Matching positions:", matching_positions)
print("Sifted Alice key:", sifted_alice_key)
print("Sifted Bob key:  ", sifted_bob_key)
print("Public QBER test positions:", test_positions)
print("Secret positions after removing public test bits:", secret_positions)
print("QBER errors:", error_count)
print("QBER:", qber)
print("QBER threshold:", qber_threshold)
print("Channel accepted:", channel_accepted)

if channel_accepted:
    print("Decision: Channel accepted. No attacker/noise detected above the threshold in this test sample.")
    print("Information reconciliation:", reconciliation_message)
    print("Final mismatches after public testing:", final_mismatches)
    print("Final Alice key:", final_alice_key)
    print("Final Bob key:  ", final_bob_key)
    print("Final keys match:", final_alice_key == final_bob_key)
else:
    print("Decision: Channel rejected. Charlie/noise detected; Alice and Bob abort and discard the key.")
    print("Information reconciliation: Not performed because the channel was rejected.")
    print("Final Alice key:", final_alice_key)
    print("Final Bob key:  ", final_bob_key)
    print("Final keys match:", final_alice_key == final_bob_key)

Index:,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99
A bit:,1,0,0,0,0,1,0,0,0,1,1,0,0,1,1,0,0,1,0,0,1,1,1,1,0,0,0,1,1,1,1,0,0,0,1,1,0,0,0,1,1,0,0,1,1,0,1,0,0,1,1,1,1,0,0,0,0,0,1,0,1,0,1,0,1,1,1,0,1,1,0,1,0,0,1,1,1,1,0,1,0,0,1,0,0,1,0,0,1,0,0,0,1,0,1,0,1,1,0,1
A basis:,d,s,d,s,d,d,d,d,d,d,d,s,s,d,s,d,d,s,d,d,s,d,d,s,d,d,s,d,s,s,d,d,d,d,d,s,d,s,d,d,s,d,s,d,d,d,s,d,s,s,s,d,d,s,s,s,d,s,d,d,d,s,s,d,d,d,s,d,s,s,d,s,d,d,d,s,d,s,d,d,s,s,s,s,d,s,d,d,d,s,d,s,s,s,d,d,s,s,s,s
A qubit:,-,0,+,0,+,-,+,+,+,-,-,0,0,-,1,+,+,1,+,+,1,-,-,1,+,+,0,-,1,1,-,+,+,+,-,1,+,0,+,-,1,+,0,-,-,+,1,+,0,1,1,-,-,0,0,0,+,0,-,+,-,0,1,+,-,-,1,+,1,1,+,1,+,+,-,1,-,1,+,-,0,0,1,0,+,1,+,+,-,0,+,0,1,0,-,+,1,1,0,1
C attack:,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y
C basis:,d,s,s,d,d,s,s,d,s,s,s,d,s,s,d,s,d,s,s,s,s,s,d,d,s,s,s,s,s,d,s,s,d,d,d,d,d,d,s,s,d,s,d,s,s,s,s,d,s,s,s,s,d,s,d,s,s,d,s,d,d,s,d,d,d,d,d,s,d,d,d,d,d,s,s,s,d,s,s,s,d,d,d,s,s,d,s,s,d,s,d,s,s,s,d,d,s,s,d,s
C bit:,1,0,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,1,1,1,1,0,1,0,0,0,0,0,1,0,1,1,0,0,1,1,0,0,0,0,1,0,0,0,1,0,1,0,0,1,1,1,1,0,0,0,1,0,0,0,1,0,0,0,1,1,0,1,1,1,0,1,0,1,0,1,1,1,0,1,0,1,0,0,0,0,1,1,1,0,0,0,1,0,1,0,1,1,1,1
C resend:,-,0,1,-,+,1,0,+,0,0,1,+,0,0,+,0,+,1,1,1,1,0,-,+,0,0,0,0,1,+,1,1,+,+,-,-,+,+,0,0,-,0,+,0,1,0,1,+,0,1,1,1,-,0,+,0,1,+,0,+,-,0,+,+,-,-,+,1,-,-,+,-,+,1,0,1,-,1,0,1,+,-,+,0,0,+,1,1,-,0,+,0,1,0,-,+,1,1,-,1
B basis:,d,d,d,s,d,s,s,s,d,s,s,s,s,s,s,d,d,s,d,d,s,d,s,s,s,d,d,s,s,s,d,s,d,s,d,d,s,d,s,d,s,d,s,s,s,d,s,s,s,d,s,d,s,d,d,s,s,d,d,d,s,d,d,s,s,d,d,s,s,s,d,d,s,s,d,d,s,s,d,d,s,d,s,s,s,d,s,s,d,d,d,d,d,d,s,d,d,s,s,s
B bit:,1,?,1,1,0,?,?,?,1,?,?,1,0,?,0,0,0,1,0,0,1,0,?,1,?,1,?,?,1,0,1,?,0,?,1,?,?,?,?,1,0,1,1,?,?,0,1,?,0,?,1,0,?,?,?,0,?,?,1,0,?,?,?,?,?,1,?,?,1,1,0,?,?,?,1,?,?,1,1,1,1,?,0,0,?,?,?,?,1,?,0,?,?,?,?,0,?,1,1,1
B raw:,1,0,1,1,0,1,0,0,1,0,1,1,0,0,0,0,0,1,0,0,1,0,0,1,0,1,1,0,1,0,1,1,0,0,1,1,1,0,0,1,0,1,1,0,1,0,1,0,0,1,1,0,0,0,0,0,1,0,1,0,0,0,0,1,1,1,0,1,1,1,0,1,0,1,1,0,0,1,1,1,1,1,0,0,0,0,1,1,1,1,0,0,1,1,0,0,1,1,1,1


Bob raw measured bits: [1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1]
Charlie attacked positions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Charlie attacked count: 100
Matching positions: [0, 2, 3, 4, 8, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 23, 25, 28, 29, 30, 32, 34, 39, 40, 41, 42, 45, 46, 48, 50, 51, 55, 58, 59, 65, 68, 69, 70, 74, 77, 78, 79, 80, 82, 83, 88, 90, 95, 97, 98, 99]
Sifted Al